In [ ]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

n, omega = sp.Symbol('n', integer=True), sp.Symbol('omega', real=True)
omega_c, omega_c1, omega_c2 = (
    sp.Symbol('omega_c', positive=True, real=True),
    sp.Symbol('omega_c1', positive=True, real=True),
    sp.Symbol('omega_c2', positive=True, real=True),
)

def get_symbolic_filter_response(filter_type):
  if filter_type == 'Low-Pass':
    return sp.Piecewise((1, sp.Abs(omega) <= omega_c), (0, True))
  elif filter_type == 'High-Pass':
    return sp.Piecewise((0, sp.Abs(omega) <= omega_c), (1, True))
  elif filter_type == 'Band-Pass':
    return sp.Piecewise((1, (sp.Abs(omega) >= omega_c1) & (sp.Abs(omega) <= omega_c2)), (0, True))
  elif filter_type == 'Band-Stop':
    return sp.Piecewise((0, (sp.Abs(omega) >= omega_c1) & (sp.Abs(omega) <= omega_c2)), (1, True))

def compute_symbolic_idft(filter_type):
  H_expr = get_symbolic_filter_response(filter_type)
  return sp.simplify(
      (1 / (2 * sp.pi))
      * sp.integrate(
          H_expr * sp.exp(sp.I * omega * n), (omega, -sp.pi, sp.pi)
      )
  )

symbolic_cache = {}

def get_cached_symbols(filter_type):
  if filter_type not in symbolic_cache:
    symbolic_cache[filter_type] = (
        get_symbolic_filter_response(filter_type),
        compute_symbolic_idft(filter_type),
    )
  return symbolic_cache[filter_type]

filter_dropdown = widgets.Dropdown(
    options=['Low-Pass', 'High-Pass', 'Band-Pass', 'Band-Stop'],
    value='Band-Stop',
    description='Filter:',
    style={'description_width': 'initial'},
)

wc_slider = widgets.FloatSlider(
    value=0.33,
    min=0.05,
    max=0.95,
    step=0.01,
    description='ωc/π (LP/HP):',
    readout_format='.2f',
    continuous_update=True,
    style={'description_width': 'initial'},
)

wc1_slider = widgets.FloatSlider(
    value=0.30,
    min=0.05,
    max=0.48,
    step=0.01,
    description='ωc1/π (BP/BS):',
    readout_format='.2f',
    continuous_update=True,
    style={'description_width': 'initial'},
)

wc2_slider = widgets.FloatSlider(
    value=0.70,
    min=0.52,
    max=0.95,
    step=0.01,
    description='ωc2/π (BP/BS):',
    readout_format='.2f',
    continuous_update=True,
    style={'description_width': 'initial'},
)

n_slider = widgets.IntSlider(
    value=20,
    min=5,
    max=50,
    step=1,
    description='Range N:',
    continuous_update=False,
    style={'description_width': 'initial'},
)

out = widgets.Output()

omega_norm_vals = np.linspace(-1.0, 1.0, 1000)
omega_vals = np.linspace(-1.0, 1.0, 1000) * np.pi

fig, axes = plt.subplots(2, 1, figsize=(10, 4))

axes[0].set_xlabel(r'Normalized Frequency \omega / \pi')
axes[0].set_ylabel(r'|H(e^j\omega)|')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_xlim(-1.0, 1.0)

axes[1].set_xlabel('Time Index n')
axes[1].set_ylabel('h[n]')
axes[1].grid(True, linestyle='--', alpha=0.6)

H_line, = axes[0].plot(omega_norm_vals, np.zeros_like(omega_norm_vals), color='b', lw=2)

n_vals = np.arange(-n_slider.value, n_slider.value + 1)
h_vals = np.zeros(len(n_vals))

stem_markers, = axes[1].plot(n_vals, h_vals, 'bo')
stem_lines = [axes[1].plot([x, x], [0, y], 'b-')[0] for x, y in zip(n_vals, h_vals)]
baseline, = axes[1].plot(n_vals, np.zeros_like(n_vals), 'r-')

axes[0].set_title('Magnitude Response |H(e^jω)| — Ideal Band-Stop Filter', fontsize=10)
axes[1].set_title('Impulse Response h[n] (Derived via IDFT) — Ideal Band-Stop Filter', fontsize=10)

plt.tight_layout()

def update_plot(filter_type, wc_norm, wc1_norm, wc2_norm, N_half):
  wc_val, wc1_val, wc2_val = wc_norm * np.pi, wc1_norm * np.pi, wc2_norm * np.pi

  H_sym, h_sym = get_cached_symbols(filter_type)

  abs_w = np.abs(omega_vals)

  if filter_type == 'Low-Pass':
    H_mag = np.where(abs_w <= wc_val, 1.0, 0.0)
  elif filter_type == 'High-Pass':
    H_mag = np.where(abs_w >= wc_val, 1.0, 0.0)
  elif filter_type == 'Band-Pass':
    H_mag = np.where((abs_w >= wc1_val) & (abs_w <= wc2_val), 1.0, 0.0)
  elif filter_type == 'Band-Stop':
    H_mag = np.where((abs_w >= wc1_val) & (abs_w <= wc2_val), 0.0, 1.0)

  H_line.set_ydata(H_mag)

  n_new = np.arange(-N_half, N_half + 1)
  h_new = np.zeros(2 * N_half + 1, dtype=float)

  h_numeric_expr = h_sym.subs(
      {omega_c: wc_val, omega_c1: wc1_val, omega_c2: wc2_val}
  )

  f_lambdified = sp.lambdify(n, h_numeric_expr, 'numpy')

  for i, val in enumerate(n_new):
    if val == 0:
      try:
        val_zero = h_numeric_expr.limit(n, 0).evalf()
        h_new[i] = float(
            val_zero.as_real_imag()[0]
            if hasattr(val_zero, 'as_real_imag')
            else val_zero
        )
      except Exception:
        fallback = h_numeric_expr.subs(n, 1e-9).evalf()
        h_new[i] = float(
            fallback.as_real_imag()[0]
            if hasattr(fallback, 'as_real_imag')
            else fallback
        )
    else:
      try:
        h_new[i] = float(np.real(f_lambdified(val)))
      except Exception:
        try:
          lim_res = h_numeric_expr.limit(n, val).evalf()
          h_new[i] = float(
              lim_res.as_real_imag()[0]
              if hasattr(lim_res, 'as_real_imag')
              else lim_res
          )
        except Exception:
          h_new[i] = 0.0

  global stem_lines, stem_markers, baseline

  for line in stem_lines:
    line.remove()

  stem_lines = [
      axes[1].plot([x, x], [0, y], 'b-')[0]
      for x, y in zip(n_new, h_new)
  ]

  stem_markers.set_data(n_new, h_new)
  baseline.set_data(n_new, np.zeros_like(n_new))

  axes[1].set_xlim(-N_half, N_half)

  axes[0].set_title(
      r'Magnitude Response |H(e^j\omega)| — Ideal ' + f'{filter_type} Filter',
      fontsize=10,
  )

  axes[1].set_title(
      f'Impulse Response h[n] (Derived via IDFT) — Ideal {filter_type} Filter',
      fontsize=10,
  )

  fig.canvas.draw_idle()

def update_symbolic_output(filter_type):
  H_sym, h_sym = get_cached_symbols(filter_type)

  with out:
    out.clear_output(wait=True)
    print(
        f'=== Selected Filter: {filter_type} ===\n'
        '1. Ideal Frequency Response H(e^jω) [SymPy]:'
    )
    display(H_sym)
    print('2. Derived Symbolic Impulse Response h[n] via IDFT [SymPy]:')
    display(h_sym)
    print('-' * 60)

def update_sliders(change):
  update_plot(
      filter_dropdown.value,
      wc_slider.value,
      wc1_slider.value,
      wc2_slider.value,
      n_slider.value,
  )

def update_filter(change):
  update_symbolic_output(filter_dropdown.value)
  update_plot(
      filter_dropdown.value,
      wc_slider.value,
      wc1_slider.value,
      wc2_slider.value,
      n_slider.value,
  )

wc_slider.observe(update_sliders, names='value')
wc1_slider.observe(update_sliders, names='value')
wc2_slider.observe(update_sliders, names='value')
n_slider.observe(update_sliders, names='value')
filter_dropdown.observe(update_filter, names='value')

display(
    widgets.VBox([
        widgets.HBox([filter_dropdown, n_slider]),
        widgets.HBox([wc_slider, wc1_slider, wc2_slider]),
        out,
        fig.canvas,
    ])
)

update_symbolic_output(filter_dropdown.value)

update_plot(
    filter_dropdown.value,
    wc_slider.value,
    wc1_slider.value,
    wc2_slider.value,
    n_slider.value,
)